# OsteoClassAI v2 · 13 Demo con detector de articulación

Exporta el detector a ONNX y lo cuantiza a int8, comprueba que la versión int8 encuentra las mismas rodillas que la original, agrega dos radiografías completas de Gornale como ejemplos (CC BY 4.0) y publica `v2/web/` en el Space de Hugging Face.

La página ahora: detecta las rodillas, recorta cada una a 1,30 × el ancho de la caja (la escala de entrenamiento) y la gradúa con el modelo D. Si la imagen ya es un recorte de la articulación, la usa tal cual.

Necesita el token de Hugging Face tipo **Write**. *Entorno de ejecución:* CPU sirve.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm imagehash h5py opencv-python-headless onnx onnxruntime onnxscript "huggingface_hub<2.0"
import sys; sys.path.insert(0, '/content/osteoclassai/v2/src')

## Exportar el detector y comparar fp32 con int8

In [ ]:
!mkdir -p outputs/det && cp -r /content/drive/MyDrive/OsteoClassAI/det/. outputs/det/
!python -m osteoclass.export_onnx --detector --ckpt $RUNS/detector/best.pt --out web/detector.onnx --quantize --check-det outputs/det --check-n 200
!cp $RUNS/convnext_tiny_sord_robust/model_int8.onnx web/
!cp web/detector_int8.onnx $RUNS/detector/
!ls -la web

Si `crop_iou_mean` es menor a 0,95 o `same_count` menor a 0,99, avise antes de publicar.

## Ejemplos de Gornale y prueba de la cadena completa con ONNX

In [ ]:
# Datos: el zip trae dos .rar (uno por experto)
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
root = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
!find "$root" -name '*.png' | wc -l   # debe dar 3300

In [ ]:
import numpy as np, pandas as pd, onnxruntime as ort, torch
from PIL import Image
from osteoclass.external import load_gray
from osteoclass.detect import letterbox, to_tensor, decode, to_original, crop_box
from osteoclass.dataset import build_transforms
aud = pd.read_csv('/content/drive/MyDrive/OsteoClassAI/external/gornale_detector/audit.csv')
rep = aud[aud.representative]
single = rep[~rep.bilateral & (rep.grade_I == 3)].sample(1, random_state=1).iloc[0]
bilat = rep[rep.bilateral & (rep.grade_I == 2)].sample(1, random_state=1).iloc[0]
det = ort.InferenceSession('web/detector_int8.onnx'); cls = ort.InferenceSession('web/model_int8.onnx')
tf = build_transforms(224)
for name, row in (('gornale_single', single), ('gornale_bilateral', bilat)):
    im = load_gray(f'{root}/{row.path}'); im.save(f'web/examples/{name}.png')
    lb, s, px, py = letterbox(np.asarray(im))
    o = det.run(['det'], {'image': to_tensor(lb)[None].numpy()})[0][0]
    knees = [to_original(k, s, px, py) for k in decode(o[0], o[1:3], o[3])]
    grades = []
    for k in knees:
        x1, y1, x2, y2 = crop_box(k['cx'], k['cy'], k['w']); side = int(round(x2 - x1))
        c = Image.new('L', (side, side), 0); c.paste(im, (-int(round(x1)), -int(round(y1))))
        x = tf(c).unsqueeze(0).numpy()
        p = sum(torch.softmax(torch.from_numpy(cls.run(['logits'], {'image': v})[0]), 1) for v in (x, x[..., ::-1].copy())) / 2
        grades.append(int(p.argmax()))
    print(f'{name}: {row.path}  experto KL {row.grade_I}  rodillas {len(knees)}  modelo KL {grades}')

## Publicar en Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
SPACE = api.whoami()['name'] + '/osteoclassai'
api.upload_folder(repo_id=SPACE, repo_type='space', folder_path='web',
                  ignore_patterns=['model.onnx', 'detector.onnx'],
                  commit_message='Demo: knee-joint detector before the classifier; full radiographs as examples')
print('Listo: https://huggingface.co/spaces/' + SPACE)